# KNN — 표본 복제 및 계보

복제 5조건 및 원천 계보 제한.

실행 전 INPUT_DIR을 수정하고 새 커널에서 순서대로 실행. 역사적 재현은 1차 실험 당시 baseline을 사용하며 후속 연구는 현재 선정 모델을 사용한다. 출력은 로컬 새 폴더에만 저장. 원본 Test 및 교수님 Test 미사용. 기존 공개 결과는 README/CSV 참고. 분리한 Notebook은 이번 정리에서 전체 학습 재실행하지 않았다.

In [ ]:
from pathlib import Path
import sys, time, json, shutil, gc, os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn import neighbors as nb
from datetime import datetime
START=Path.cwd().resolve()
FT=next((p for p in (START,*START.parents) if (p/"00_Common"/"prepare.py").is_file()),None)
if FT is None: raise FileNotFoundError("Open this notebook within cloned 07_fine_tuning")
sys.path.insert(0,str(FT/"00_Common"))
from prepare import prepare, metrics, evaluate_followup
INPUT_DIR=Path.home()/"ml_fine_tuning_inputs" # set to approved private input directory
OUTPUT_DIR=Path.home()/"ml_fine_tuning_runs"/("KNN_historical_"+datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
CURRENT_SVC=None # followup SVC: set trusted C2_G2 joblib path
state=prepare("KNN", INPUT_DIR, OUTPUT_DIR, mode="historical",current_svc=CURRENT_SVC)
globals().update(state)
def stats(pred,base,y=y):
    return metrics(y,np.asarray(pred),np.asarray(base))
knn=model; tr=knn._fit_X; ty=knn._y; kbase=base
print("Local output:",root)

In [ ]:
def vote(d,ids,k,alpha,train_y=ty):
    ds=d[:,:k]; ls=train_y[ids[:,:k]]
    zero=ds==0
    weights=np.power(np.maximum(ds,1e-30),-alpha)
    weights=np.where(zero.any(axis=1)[:,None],zero,weights)
    scores=np.zeros((len(ds),16))
    np.add.at(scores,(np.arange(len(ds))[:,None],ls),weights)
    return np.argmax(scores,axis=1)


In [ ]:
import sklearn.neighbors as nb
from final_knn_normal import sample_balanced,select_extra,BASE_FRACTION,SEED,WEAK_LABELS,ADDITIONAL_COUNT
rawh=np.concatenate([wr[w]['X'] for w in fb.W]); yh=np.concatenate([wr[w]['y'] for w in fb.W])
ix=fb.balanced_nested_indices(np.repeat(fb.W,640),yh,seed=fb.HAND_SEED)
qidx=ix[25]; ridx=np.setdiff1d(ix[50],ix[25],assume_unique=True)
sel=sample_balanced(off['y_train'],fraction=BASE_FRACTION,seed=SEED)
extra=select_extra(off['y_train'],sel,ADDITIONAL_COUNT,[labels.index(s) for s in WEAK_LABELS],seed=SEED+142)
# source group ID from actual official split + augmented lineages; existing official archive includes original source groups
groups=np.asarray(off['train_group_ids'],dtype=np.int64)
# use unique source groups, avoid collision with hand IDs (negative)
hands=-1-np.arange(2560,dtype=np.int64)
lineage=np.concatenate([groups,groups[sel],groups[extra],hands,hands,hands[qidx],hands[ridx]])
assert len(lineage)==len(ty)
assert np.array_equal(ty[107520:110080],yh) and np.array_equal(ty[110080:112640],yh)
assert np.array_equal(tr[112640:113280],tr[107520+qidx]) and np.array_equal(tr[113280:113920],tr[110080+ridx])
print('repeat rows and lineage check passed',len(np.unique(lineage)))
np.save(root/'03_KNN/train_lineage_ids.npy',lineage)
# repeat ablations
rrows=[]
rpreds={}
for name,ixr in [('R00',np.arange(len(ty))),('R01_remove_all',np.arange(112640)),('R02_original_only',np.arange(113280)),('R03_augmented_only',np.r_[np.arange(112640),np.arange(113280,113920)])]:
 t=time.perf_counter()
 if name=='R00': pred=kbase
 else:
  mm=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='euclidean',algorithm='brute',n_jobs=2).fit(tr[ixr],ty[ixr])
  pred=mm.predict(Xv);del mm
 rpreds[name]=pred
 rrows.append({'condition':name,'train_rows':len(ixr),'batch_eval_s':time.perf_counter()-t,**stats(pred,kbase)})
print(pd.DataFrame(rrows)[['condition','train_rows','correct','macro_f1','corrected','regressed']])

In [ ]:
# R04: select distinct original source-group neighbors; adaptive expansion if needed.
t=time.perf_counter()
dl,il=knn.kneighbors(Xv,n_neighbors=32)
chosen_d=[];chosen_i=[]
for j,(drow,irow) in enumerate(zip(dl,il)):
 seen=set();pairs=[]
 for d,i in zip(drow,irow):
  g0=int(lineage[i])
  if g0 not in seen: pairs.append((d,i));seen.add(g0)
  if len(pairs)==3:break
 if len(pairs)<3:
  dd,ii=knn.kneighbors(Xv[j:j+1],n_neighbors=128)
  seen=set();pairs=[]
  for d,i in zip(dd[0],ii[0]):
   g0=int(lineage[i])
   if g0 not in seen:pairs.append((d,i));seen.add(g0)
   if len(pairs)==3:break
 assert len(pairs)==3
 chosen_d.append([z[0] for z in pairs]);chosen_i.append([z[1] for z in pairs])
pred=vote(np.asarray(chosen_d),np.asarray(chosen_i),3,1)
rpreds['R04_lineage_cap']=pred
rrows.append({'condition':'R04_lineage_cap','train_rows':len(ty),'batch_eval_s':time.perf_counter()-t,**stats(pred,kbase)})
pd.DataFrame(rrows).to_csv(root/'03_KNN/EXP-FT-KNN-003_handmade.csv',index=False)
pd.DataFrame({'true':y,**rpreds}).to_csv(root/'03_KNN/EXP-FT-KNN-003_predictions.csv',index=False)
print(pd.DataFrame(rrows)[['condition','correct','macro_precision','macro_f1','corrected','regressed']])